# Comprobación de migración: estados con tres theta

Predicción sin recalibrar parámetros y sin cambiar las siete ecuaciones actuales. Incluye biomasa muerta, glicerol, mortalidad y mantenimiento atenuado tal como están implementados. Datos depurados LAB004–009 de `recalibration`; idénticas entradas para los tres vectores.

**`theta_defaults` conserva once parámetros de `theta_full` y sustituye seis heredados por defaults**, siguiendo los notebooks históricos y la v3. No es el vector de diecisiete defaults. `theta_new` es el reajuste anterior, no el ajuste con rendimientos restringidos. Ningún límite nuevo se aplica a estos vectores.

Los parámetros, los insumos de compatibilidad y las funciones necesarias viven dentro de este proyecto. No se importa código del repositorio anterior. Las figuras permanecen en el EXECUTED.

In [ ]:
from pathlib import Path
from datetime import datetime
from dataclasses import replace
import sys, json, hashlib, time, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, JSON
ROOT=next(p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents)
          if (p/'data/sources.csv').is_file() and (p/'src/wine_fermentation').is_dir())
if str(ROOT/'src') not in sys.path:sys.path.insert(0,str(ROOT/'src'))
sys.dont_write_bytecode=True
from wine_fermentation import current_model as base
assert Path(sys.prefix).name.lower()=='fermentation',sys.executable
CASES=('theta_full','theta_new','theta_defaults')
COLORS={'theta_full':'#0072B2','theta_new':'#D55E00','theta_defaults':'#009E73'}
STYLES={'theta_full':'-','theta_new':'--','theta_defaults':':'}
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi':100,'font.size':9,'font.family':'DejaVu Sans'})
pd.set_option('display.max_columns',30)
THETAS={name:json.loads((ROOT/f'configs/theta/{name}.json').read_text()) for name in CASES}
assert all(set(theta)==set(base.FULL17) for theta in THETAS.values())
TRACE={'python':sys.executable,'numpy':np.__version__,'pandas':pd.__version__}
START=time.perf_counter()
print('Entorno:',sys.executable)
print('Raíz independiente:',ROOT)


## 1. Parámetros congelados y significado de los rendimientos

In [ ]:
parameters=pd.DataFrame(THETAS).reindex(base.FULL17)
display(parameters)
display(pd.DataFrame([dict(theta=name,Y_E_G=t['betaG0']/t['qEG'],Y_E_F=t['betaF0']/t['qEF']) for name,t in THETAS.items()]))
assert {p for p in base.FULL17 if THETAS['theta_full'][p]!=THETAS['theta_defaults'][p]}=={'sN','qXG','qXF','sG','sF','m0'}
OUT=ROOT/'results/compatibility_states';OUT.mkdir(parents=True,exist_ok=True)
parameters.to_csv(OUT/'parameters.csv')


## 2. Datos y condiciones iniciales comunes

In [ ]:
d=pd.read_csv(ROOT/'data/compatibility/chemistry/observations_cleaned_lab004_009.csv')
LABS=tuple(f'LAB{i:03}' for i in range(4,10))
assert set(d.batch)==set(LABS)
for state,column in base.STATE_COLUMNS.items():
    d[column]=pd.to_numeric(d[column],errors='coerce')
    d.loc[d[column]<0,column]=0.  # misma regla histórica
for lab,expected in [('LAB004',1.044),('LAB005',1.5576),('LAB006',.9195)]:
    assert np.isclose(d.loc[d.batch.eq(lab)&d.sample_id.eq(f'ING26-{lab}-04'),'X_viable_kg_m3'].iloc[0],expected)
original_batches=base.make_batches(d)
clocks=pd.read_csv(ROOT/'data/compatibility/chemistry/clocks.csv').drop_duplicates().set_index('batch').t0
pulses={}
for lab in LABS:
    sheet=pd.read_excel(ROOT/'data/cleaned/natural/lab004_012/mosto_natural_depurado.xlsx',sheet_name=lab)
    event=sheet[sheet['pulso real'].notna()]
    assert len(event)==1 and np.isclose(event.pulso_nut.iloc[0],80.)
    pulses[lab]=float(event['pulso real'].iloc[0])
display(pd.DataFrame([dict(lab=b.batch,**b.initials,pulse_h=pulses[b.batch],dose_N_g_L=.08) for b in original_batches]))
display(Markdown('Los horarios de pulso propuestos en el Excel depurado son supuestos de esa revisión; se mantienen para comparar theta y no se convierten en horarios de ejecución confirmados.'))


## 3. Temperatura: mismo filtrado de la revisión química

In [ ]:
from scipy.signal import butter,sosfiltfilt
profiles={};thermal_audit=[];batches=[]
for lab in LABS:
    paths=sorted(p for p in (ROOT/'data/original/natural/process_logs').rglob('*.csv')
                 if p.name.lower().startswith('temp') and re.search(rf'lab0*{int(lab[3:])}(?:\D|$)',p.name,re.I))
    fragments=[]
    for path in paths:
        part=pd.read_csv(path,usecols=['timestamp','T','SP'])
        part['timestamp']=pd.to_datetime(part.timestamp,errors='coerce')
        for col in ('T','SP'):part[col]=pd.to_numeric(part[col],errors='coerce')
        fragments.append(part.dropna(subset=['timestamp']))
    raw=pd.concat(fragments).sort_values('timestamp',kind='stable')
    duplicate=raw[raw.timestamp.duplicated(keep=False)]
    if len(duplicate):assert duplicate.groupby('timestamp')[['T','SP']].nunique().le(1).all().all()
    raw=raw.drop_duplicates('timestamp');t0=pd.Timestamp(clocks[lab])
    raw['t_h']=(raw.timestamp-t0).dt.total_seconds()/3600.
    raw=raw[raw.t_h.ge(0)].dropna(subset=['T'])
    profile=raw.set_index('timestamp')[['T','SP']].resample('30min').median().dropna(subset=['T']).reset_index()
    profile['t_h']=(profile.timestamp-t0).dt.total_seconds()/3600.+.25
    profile=profile[profile.t_h.between(0,raw.t_h.max())].copy()
    gap=profile.t_h.diff().gt(.500001);step=profile.SP.diff().abs().gt(.5)
    profile['segment']=(gap|step).cumsum();profile['filtered']=np.nan
    sos=butter(2,1/4.,fs=2.,output='sos')
    for _,group in profile.groupby('segment'):
        smooth=group['T'].rolling(3,center=True,min_periods=1).median().to_numpy()
        profile.loc[group.index,'filtered']=sosfiltfilt(sos,smooth) if len(smooth)>9 else smooth
    profiles[lab]=profile
    old=next(b for b in original_batches if b.batch==lab)
    horizon=float(old.time[-1]);knots=np.arange(0,horizon+1e-9,2.)
    switches=profile.loc[profile.segment.diff().fillna(0).ne(0),'t_h'].to_numpy()
    knots=np.r_[knots,switches,switches-.5]
    grid=np.unique(np.r_[old.time,knots[(knots>=0)&(knots<=horizon)]])
    observations={state:np.full(len(grid),np.nan) for state in base.STATE_NAMES}
    index=np.searchsorted(grid,old.time)
    for state in base.STATE_NAMES:observations[state][index]=old.observations[state]
    batches.append(replace(old,time=grid,temperature_c=np.interp(grid,profile.t_h,profile.filtered),
        pulses={**old.pulses,'N':((pulses[lab],.08),)},observations=observations))
    thermal_audit.append(dict(lab=lab,first_sensor_h=profile.t_h.min(),last_sensor_h=profile.t_h.max(),horizon_h=horizon))
display(pd.DataFrame(thermal_audit))
fig,axes=plt.subplots(2,3,figsize=(15,6),layout='constrained')
for ax,b in zip(axes.flat,batches):
    p=profiles[b.batch];ax.plot(p.t_h,p['T'],color='.7',label='Mediana 30 min')
    ax.plot(b.time,b.temperature_c,label='Entrada utilizada');ax.set_title(b.batch);ax.set_ylabel('°C');ax.set_xlabel('Tiempo histórico [h]')
axes.flat[0].legend();display(fig);plt.close(fig)


## 4. Comprobación contra referencias del RHS y simulador originales

In [ ]:
fixture=np.load(ROOT/'tests/reference/original_core_predictions.npz')
checks=[]
for case,theta in THETAS.items():
    rates=[]
    for temperature in (16.,20.,24.,32.):
        bt=base.BatchData('natural','CHECK',np.array([0.,72.]),np.array([temperature]*2),{}, {},dict(zip(base.STATE_NAMES,fixture['states'][0])))
        rates.extend(base.rhs(0.,state,theta,bt) for state in fixture['states'])
    rhs_error=float(np.max(abs(np.asarray(rates)-fixture[case+'_rhs'])))
    grid=fixture['time']
    bt=base.BatchData('natural','CHECK',grid,np.interp(grid,[0.,24.,48.,72.],[16.,21.,18.,24.]),
        {'N':((23.5,.08),(48.25,.04))},{},dict(zip(base.STATE_NAMES,fixture['states'][0])))
    current=base.simulate(bt,theta,grid)
    trajectory_error=float(np.max(abs(current.to_numpy()-fixture[case+'_trajectory'])))
    assert rhs_error<1e-13 and trajectory_error<1e-10
    checks.append(dict(theta=case,rhs_max_abs=rhs_error,trajectory_max_abs=trajectory_error))
display(pd.DataFrame(checks));pd.DataFrame(checks).to_csv(OUT/'original_model_parity.csv',index=False)


## 5. Predicción sin reajustar y errores por estado

In [ ]:
predictions={};metrics=[];trajectories=[]
for case,theta in THETAS.items():
    for batch in batches:
        pulse=batch.pulses['N'][0][0]
        times=np.unique(np.r_[np.arange(0.,batch.time[-1],.5),batch.time,batch.time[-1],pulse])
        cr=base.simulate(batch,theta,times)
        assert cr is not None and np.isfinite(cr.to_numpy()).all()
        cr['GF']=cr.G+cr.F;predictions[case,batch.batch]=cr
        trajectories.append(cr.rename_axis('t_h').reset_index().assign(theta=case,lab=batch.batch))
        for state,column in {**base.STATE_COLUMNS,'GF':'S_GF_g_l'}.items():
            group=d[d.batch.eq(batch.batch)].sort_values('time_h')
            points=group.loc[group.time_h.gt(0.),['time_h',column]].dropna()
            if points.empty:continue
            p=np.interp(points.time_h,cr.index,cr[state]);y=points[column].to_numpy()
            err=p-y
            metrics.append(dict(theta=case,lab=batch.batch,state=state,n=len(y),RMSE=float(np.sqrt(np.mean(err**2))),
                MAE=float(np.mean(abs(err))),bias=float(err.mean()),final_predicted=float(cr[state].iloc[-1])))
metrics=pd.DataFrame(metrics)
pd.concat(trajectories,ignore_index=True).to_csv(OUT/'state_trajectories.csv',index=False)
metrics.to_csv(OUT/'metrics_by_lab_state.csv',index=False)
display(metrics.pivot_table(index=['lab','state'],columns='theta',values='RMSE').round(4))


## 6. Curvas completas y observaciones

In [ ]:
for state,label in [('X','Biomasa viable'),('N','YAN'),('G','Glucosa'),('F','Fructosa'),('GF','Glucosa + fructosa'),('E','Etanol'),('Xd','Biomasa muerta'),('Gly','Glicerol')]:
    fig,axes=plt.subplots(2,3,figsize=(15,7),sharex=False,layout='constrained')
    mult=1000. if state=='N' else 1.
    column='S_GF_g_l' if state=='GF' else base.STATE_COLUMNS[state]
    for ax,batch in zip(axes.flat,batches):
        group=d[d.batch.eq(batch.batch)].sort_values('time_h')
        real=group[group.time_h.gt(0.)].dropna(subset=[column])
        ax.scatter(real.time_h,real[column]*mult,color='black',s=16,label='Observaciones')
        initial=batch.initials['G']+batch.initials['F'] if state=='GF' else batch.initials[state]
        ax.scatter([0.],[initial*mult],facecolors='white',edgecolors='black',s=30,label='Entrada inicial',zorder=5)
        for case in CASES:
            cr=predictions[case,batch.batch]
            ax.plot(cr.index,cr[state]*mult,color=COLORS[case],linestyle=STYLES[case],label=case)
        ax.axvline(pulses[batch.batch],color='.5',linestyle=':',linewidth=1.)
        ax.set_title(batch.batch);ax.set_xlabel('Tiempo histórico [h]');ax.set_ylabel('mg N/L' if state=='N' else 'g/L')
    fig.suptitle(label+' · modelo actual sin recalibrar')
    handles,labels=axes.flat[0].get_legend_handles_labels();fig.legend(handles,labels,loc='outside lower center',ncol=5)
    display(fig);plt.close(fig)


## 7. Alcance de la comprobación

In [ ]:
TRACE.update({'theta_files':{name:hashlib.sha256((ROOT/f'configs/theta/{name}.json').read_bytes()).hexdigest() for name in CASES},
    'equation_states':list(base.STATE_NAMES),'parameters_refitted':False,'elapsed_s':time.perf_counter()-START,
    'parity':checks,'observations_exclude_initial_from_metrics':True})
display(JSON(TRACE,expanded=False))
display(Markdown('La paridad verifica el traslado de ecuaciones e integración. Las diferencias de RMSE entre vectores son resultados de predicción sobre estas entradas; no demuestran por sí mismas identificabilidad ni una mejora generalizable. El modelo y sus restricciones físicas históricas permanecen intactos.'))
